# Практика · Автоенкодери й VAE

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає **суддю**, ще двох суддів для порівняння, чотири генеративні
> моделі на трьох зернах кожну і вісімнадцять автоенкодерів різної ширини — разом
> **тридцять три навчання**. Заміряно на чотирьох ядрах без відеокарти:
> **близько чотирьох хвилин** (239 с у чистому прогоні; на завантаженій машині
> доходило до шести). Рахунок іде в один потік — так і швидше на дрібних тензорах,
> і відтворюваніше.

Тут кожне число лекції отримує свій рядок друку. Порядок такий:

1. **Шумова підлога** — наскільки точним відновлення взагалі може бути.
2. **Суддя** — окремий класифікатор фігур; перевіряємо, що задача для нього тривіальна.
3. **Калібрування метрики** — головний крок. Прогонимо кандидатів у метрики через
   вісім завідомо поганих наборів і викидаємо ті, що не ловлять поломку.
4. **Автоенкодер проти VAE** — чотири моделі, три зерна, пʼять чисел на кожну.
5. **Суддя різної гостроти** — доказ, що впевненість є властивістю судді, а не зразків.
6. **Ширина вузького місця** — де латент перестає допомагати.
7. **Де ми беремо z** — чому вибірка з N(0, 1) в автоенкодера потрапляє в порожнечу.
8. **Інтерполяція** — замір, який **не** підтвердив підручникову тезу.
9. **Карта латентного простору** — що там групується насправді.
10. **Репараметризація** — чому `mu + sigma·eps` можна, а просто насемплити не можна.

In [ ]:
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Один потік, а не чотири. Мережі тут крихітні, і чотири потоки більше часу
# домовляються між собою, ніж рахують. Друга причина важливіша за швидкість:
# під кількома потоками float-суми йдуть в іншому порядку, і числа пливуть.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## 1 · Датасет: ті самі шість фігур

Це наскрізний набір усього блоку 8 — той самий генератор, що в
[темі 16](../16-self-supervised/lecture.html). Шість фігур 28 на 28, центр гуляє
на 5 пікселів у будь-який бік, радіус від 5 до 8, зверху гаусів шум 0.06.

Шум малий навмисне: у цій темі ми міряємо, наскільки добре мережа **відновлює**
картинку, і зайвий шум просто підняв би всім однакову похибку.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=5, noise=0.06, center=None, radius=None):
    # Малює одну фігуру як масив 28 на 28 зі значеннями 0..1.
    # center і radius можна задати явно — тоді генератор випадкових чисел не
    # потрібен узагалі. Саме так ми зробимо «еталонну» фігуру без шуму й зсуву.
    image = np.zeros((size, size), dtype=np.float32)
    if center is None:
        center_y = size / 2 + rng.integers(-jitter, jitter + 1)
        center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    else:
        center_y, center_x = center
    if radius is None:
        radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    if noise > 0:
        image = image + rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    # Повертає (count, 1, 28, 28) і (count,). Класів шість, порівну.
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6                       # рівно по шостій частині кожного класу
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


rng = np.random.default_rng(42)
x_train, y_train = make_dataset(1800, rng)      # на цьому вчиться все
x_test, y_test = make_dataset(600, rng)         # на цьому міряємо відновлення
x_bank, y_bank = make_dataset(600, rng)         # «полиця справжніх» для відстаней

print("навчальні :", tuple(x_train.shape))
print("тестові   :", tuple(x_test.shape))
print("полиця    :", tuple(x_bank.shape))
print("класів    :", len(SHAPE_NAMES), "-> рівень вгадування %.4f" % (1 / 6))

In [ ]:
import matplotlib.pyplot as plt

# «еталонні» фігури: рівно по центру, без шуму — тільки геометрія
clean_shapes = [draw_shape(kind, None, center=(13.5, 13.5), radius=7, noise=0.0)
                for kind in range(6)]

figure, axes = plt.subplots(2, 6, figsize=(9, 3.3))
for kind in range(6):
    axes[0, kind].imshow(clean_shapes[kind], cmap="gray", vmin=0, vmax=1)
    axes[0, kind].set_title(SHAPE_NAMES[kind], fontsize=9)
    axes[1, kind].imshow(x_train[kind, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, kind].set_title("як у наборі", fontsize=8)
    axes[0, kind].axis("off")
    axes[1, kind].axis("off")
plt.tight_layout()
plt.show()
print("верхній ряд — чиста геометрія, нижній — те, що бачить мережа")

## 2 · Шумова підлога: наскільки точним відновлення взагалі може бути

Перш ніж радіти чи засмучуватись з похибки відновлення, треба знати, куди вона
взагалі може впасти. Наші зображення зашумлені, і **шум невідновний за побудовою**:
навіть якби мережа ідеально вгадала форму, центр і радіус, вона все одно не вгадає
конкретний візерунок шуму.

Порахуємо цю межу прямо: згенеруємо чисті фігури, додамо той самий шум і порівняємо.

In [ ]:
geometry_rng = np.random.default_rng(2024)
clean = np.zeros((600, 1, 28, 28), dtype=np.float32)
noisy = np.zeros_like(clean)

for i in range(600):
    # ті самі межі, що в draw_shape: центр гуляє на 5, радіус від 5 до 8
    center_y = 13.5 + geometry_rng.integers(-5, 6)
    center_x = 13.5 + geometry_rng.integers(-5, 6)
    radius = int(geometry_rng.integers(5, 9))
    clean[i, 0] = draw_shape(i % 6, None, center=(center_y, center_x),
                             radius=radius, noise=0.0)
    speckle = geometry_rng.normal(0, 0.06, (28, 28)).astype(np.float32)
    noisy[i, 0] = np.clip(clean[i, 0] + speckle, 0, 1)

noise_floor = float(((noisy - clean) ** 2).mean())
print("шумова підлога (ідеальне відновлення форми): %.5f" % noise_floor)
print("нижче цього числа не спуститься жодна модель — там уже не форма, а шум")

## 3 · Суддя

Генерацію не можна оцінити «на око» в матеріалі, де кожне число мусить надрукувати
програма. Тому блок 8 уводить **суддю** — окремий класифікатор шести фігур,
навчений на справжніх даних і більше ні на чому.

Мережа звичайна: два згорткові блоки й лінійна голова. Важлива тут не архітектура,
а перевірка: **задача для судді має бути тривіальною**. Якщо він плутається на
справжніх фігурах, його думка про згенеровані нічого не варта.

In [ ]:
class Judge(nn.Module):
    # Класифікатор шести фігур: Conv -> ReLU -> Pool двічі, потім лінійна голова.

    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten())
        self.head = nn.Linear(32 * 7 * 7, 6)

    def forward(self, x):
        return self.head(self.body(x))


def train_judge(epochs=40, batch=300, lr=2e-3):
    # Навчає суддю на справжніх фігурах. Зерно фіксоване — суддя один на всіх.
    torch.manual_seed(0)
    judge_model = Judge()
    optimizer = torch.optim.AdamW(judge_model.parameters(), lr=lr)
    shuffler = torch.Generator().manual_seed(0)
    for _ in range(epochs):
        order = torch.randperm(1800, generator=shuffler)
        for start in range(0, 1800, batch):
            batch_idx = order[start:start + batch]
            loss = F.cross_entropy(judge_model(x_train[batch_idx]), y_train[batch_idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    judge_model.eval()
    return judge_model


started = time.perf_counter()
judge = train_judge()
with torch.no_grad():
    judge_accuracy = (judge(x_test).argmax(1) == y_test).float().mean().item()
    judge_confidence_real = F.softmax(judge(x_test), 1).max(1).values.mean().item()

print("точність судді на справжніх фігурах : %.4f" % judge_accuracy)
print("середня впевненість на них же       : %.4f" % judge_confidence_real)
print("навчання судді зайняло %.0f с" % (time.perf_counter() - started))

## 4 · Чотири кандидати в метрики

Далі нам треба одним числом сказати, наскільки добре модель генерує. Кандидатів
чотири, і всі чотири виглядають розумно.

**Впевненість судді.** Проганяємо згенеровані зображення крізь суддю й дивимось,
наскільки він упевнений: середнє від найбільшої ймовірності. Логіка: якщо це
справді схоже на фігуру, суддя впевнений; якщо ні — сумнівається.

**Покриття класів.** Скільки з шести фігур узагалі трапилось у вибірці. Клас
рахуємо, якщо він набрав хоча б **1 %** — шість зразків із шестисот. Інакше одна
випадкова картинка дає цілий клас.

**Відстань «зразок -> найближча справжня».** Для кожного згенерованого зображення
шукаємо найсхожішу справжню фігуру з полиці й беремо відстань до неї. Логіка:
гарний зразок має бути схожий хоч на щось справжнє.

**Відстань «справжня -> найближчий зразок».** Дзеркальна до попередньої: для кожної
справжньої фігури шукаємо найсхожіший згенерований зразок. Логіка інша: навіть якщо
всі зразки гарні, вони мусять покривати **все** розмаїття справжніх фігур, а не два
його куточки.

In [ ]:
@torch.no_grad()
def measure(images, which_judge=None):
    # Усі чотири кандидати в метрики одним проходом.
    which_judge = which_judge or judge
    probabilities = F.softmax(which_judge(images), 1)
    confidence = probabilities.max(1).values.mean().item()
    counts = torch.bincount(probabilities.argmax(1), minlength=6).numpy()

    # матриця відстаней «кожен зразок на кожну справжню фігуру»
    flat_generated = images.view(images.shape[0], -1)
    flat_real = x_bank.view(x_bank.shape[0], -1)
    distances = torch.cdist(flat_generated, flat_real)

    return dict(
        conf=confidence,
        cnt=counts.tolist(),
        # клас рахуємо, якщо він набрав хоч 1 % вибірки
        cov=int((counts >= images.shape[0] // 100).sum()),
        # від зразка до найближчої справжньої: «чи схоже це на фігуру»
        near_gen=distances.min(1).values.mean().item(),
        # від справжньої до найближчого зразка: «чи всі фігури хтось намалював»
        near_real=distances.min(0).values.mean().item())


probe = measure(x_test)
print("на справжніх тестових фігурах:")
print("  впевненість судді             : %.4f" % probe["conf"])
print("  класів із шести               : %d" % probe["cov"])
print("  зразок -> найближча справжня  : %.4f" % probe["near_gen"])
print("  справжня -> найближчий зразок : %.4f" % probe["near_real"])

## 5 · Калібрування: перш ніж міряти, перевір вимірювач

Це головний крок теми, і пропускати його не можна. Метрика генерації — не
термометр, перевірений на заводі. Це наш власний саморобний прилад, і поки він не
показав правильно на **завідомо поганому**, вірити його показанням на невідомому
не можна.

Зберемо вісім наборів, кожен по 600 зображень, і кожен зіпсований по-своєму:

- **справжні, інша вибірка** — контроль: тут метрика має показати «добре»;
- **лише кола** — повний колапс: один клас із шести, зате всі зразки бездоганні;
- **кола + квадрати** — колапс мʼякший, два класи;
- **розмита каша** — справжні фігури, розмиті вікном 9 на 9: форма ще вгадується,
  але це вже не фігура;
- **середнє всіх фігур** — одна й та сама сіра пляма 600 разів;
- **усе чорне**, **усе біле**, **чистий шум** — очевидне сміття.

In [ ]:
bad_rng = np.random.default_rng(7)


def batch_of(kinds, count=600):
    # Вибірка, у якій трапляються тільки перелічені класи.
    return torch.from_numpy(np.stack(
        [draw_shape(kinds[i % len(kinds)], bad_rng)[None] for i in range(count)]))


real_other, _ = make_dataset(600, np.random.default_rng(99))
only_circles = batch_of([0])
circles_squares = batch_of([0, 1])
all_six = batch_of([0, 1, 2, 3, 4, 5])
# розмиття вікном 9 на 9: краї доповнюємо копією крайнього пікселя, щоб не було рамки
blurred = F.avg_pool2d(F.pad(all_six, (4, 4, 4, 4), mode="replicate"), 9, stride=1)
mean_shape = x_train.mean(0, keepdim=True).repeat(600, 1, 1, 1)
all_black = torch.zeros(600, 1, 28, 28)
all_white = torch.ones(600, 1, 28, 28)
torch.manual_seed(3)
pure_noise = torch.rand(600, 1, 28, 28)

calibration_sets = [
    ("справжні, інша вибірка", real_other),
    ("лише кола", only_circles),
    ("кола + квадрати", circles_squares),
    ("розмита каша", blurred),
    ("середнє всіх фігур", mean_shape),
    ("усе чорне", all_black),
    ("чистий шум", pure_noise),
    ("усе біле", all_white),
]

calibration = []
print("%-24s %10s %8s %14s %15s" %
      ("що подаємо", "впевн.", "класів", "зразок->спр.", "спр.->зразок"))
print("-" * 76)
for name, images in calibration_sets:
    row = measure(images)
    row["name"] = name
    calibration.append(row)
    print("%-24s %10.4f %8d %14.4f %15.4f" %
          (name, row["conf"], row["cov"], row["near_gen"], row["near_real"]))

### Читаємо таблицю — і викидаємо дві метрики з чотирьох

**Впевненість судді провалилась повністю.** «Усе біле» дає найбільше число в
таблиці — більше, ніж справжні фігури. «Усе чорне» теж вище за справжні. «Лише
кола», тобто повний колапс мод, майже дорівнює справжнім.

Причина не в поганому судді. Суддя навчений відповідати на питання **«яка це з
шести фігур»**, а не «чи це взагалі фігура». Сьомого варіанта «нічого з
переліченого» в нього немає й бути не може: softmax завжди сумується в одиницю.
Біле полотно — просто точка, дуже далека від межі між класами, і саме тому суддя
такий у ній упевнений.

**Відстань «зразок -> найближча справжня» теж провалилась.** У «лише кола» вона
менша, ніж у справжніх фігур, а в «кола + квадрати» — найменша в усій таблиці. Це
логічно й тому небезпечно: якщо малювати тільки кола, кожне намальоване коло
справді дуже схоже на якесь справжнє коло з полиці. Метрика міряє **якість окремого
зразка** і за побудовою сліпа до того, що зразки всі однакові.

**Покриття класів ловить колапс** — 1 і 2 замість шести. Але воно сліпе до якості:
подивись на «чистий шум».

**Відстань «справжня -> найближчий зразок» ловить решту.** Вона велика і в колапсі
(для справжнього кільця немає жодного схожого кола), і в каші, і в сміттях.

Отже робочою метрикою теми буде **пара**: покриття класів разом із відстанню
«справжня -> найближчий зразок». Одне число тут не працює — і це не наша
слабкість, а причина, з якої для генерації взагалі довелося вигадувати окремі
метрики.

In [ ]:
# перевіримо явно вимогу калібрування: чи стоїть кожен зіпсований набір гірше
# за контроль хоча б за одним числом робочої пари
reference = calibration[0]
print("контроль (справжні): класів %d, справжня->зразок %.4f" %
      (reference["cov"], reference["near_real"]))
print()
for row in calibration[1:]:
    worse_by_coverage = row["cov"] < reference["cov"]
    worse_by_recall = row["near_real"] > reference["near_real"]
    verdict = "ловить" if (worse_by_coverage or worse_by_recall) else "НЕ ЛОВИТЬ"
    print("%-24s класів %d (%-9s) справжня->зразок %7.4f (%-9s) -> %s" %
          (row["name"], row["cov"], "гірше" if worse_by_coverage else "не гірше",
           row["near_real"], "гірше" if worse_by_recall else "не гірше", verdict))

## 6 · Автоенкодер і VAE: один кістяк, дві втрати

Тепер самі моделі. Кістяк у них **однаковий**, і це принципово: якби ми порівнювали
різні архітектури, різницю можна було б списати на що завгодно.

**Енкодер**: дві згортки з кроком 2 (28 на 28, далі 14 на 14, далі 7 на 7), потім
лінійний шар у латент. **Декодер**: дзеркало — лінійний шар, потім дві транспоновані
згортки назад до 28 на 28, на виході сигмоїда, бо пікселі лежать у проміжку від
нуля до одиниці.

Різниця рівно у двох місцях:

- автоенкодер видає з енкодера **латент чисел**, VAE — **удвічі більше**: по одному
  середньому `mu` і одному логарифму дисперсії `logvar` на кожен вимір;
- у VAE до втрати відновлення додається друга половина — розбіжність Кульбака —
  Лейблера між хмаркою, яку видав енкодер, і стандартним нормальним розподілом.

In [ ]:
CHANNELS_1, CHANNELS_2 = 8, 16
FEATURES = CHANNELS_2 * 7 * 7          # 784 числа після двох згорток


class Encoder(nn.Module):
    # 28 на 28 -> latent чисел (для VAE удвічі більше: середні й логарифми дисперсій)

    def __init__(self, latent, outputs=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, CHANNELS_1, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(CHANNELS_1, CHANNELS_2, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten())
        self.out = nn.Linear(FEATURES, latent * outputs)

    def forward(self, x):
        return self.out(self.conv(x))


class Decoder(nn.Module):
    # latent чисел -> 28 на 28. Сигмоїда на виході тримає пікселі в межах 0..1.

    def __init__(self, latent):
        super().__init__()
        self.lin = nn.Linear(latent, FEATURES)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(CHANNELS_2, CHANNELS_1, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(CHANNELS_1, 1, 4, stride=2, padding=1), nn.Sigmoid())

    def forward(self, z):
        return self.deconv(F.relu(self.lin(z)).view(-1, CHANNELS_2, 7, 7))


class Autoencoder(nn.Module):
    def __init__(self, latent=8):
        super().__init__()
        self.enc = Encoder(latent, 1)
        self.dec = Decoder(latent)
        self.latent = latent

    def code(self, x):
        return self.enc(x)


class VariationalAutoencoder(nn.Module):
    def __init__(self, latent=8):
        super().__init__()
        self.enc = Encoder(latent, 2)
        self.dec = Decoder(latent)
        self.latent = latent

    def encode(self, x):
        # Повертає центр хмарки і логарифм її дисперсії — по одному на вимір.
        both = self.enc(x)
        return both[:, :self.latent], both[:, self.latent:]

    def code(self, x):
        # для відновлення беремо центр хмарки, без випадковості
        return self.encode(x)[0]


print("параметрів в автоенкодері з латентом 8:",
      sum(p.numel() for p in Autoencoder(8).parameters()))
print("параметрів у VAE з латентом 8         :",
      sum(p.numel() for p in VariationalAutoencoder(8).parameters()))
print("різниця — це рівно другий вихід енкодера:", 8 * FEATURES + 8)

Тепер навчання. Обидві моделі йдуть однаковим кодом; `beta` дорівнює нулю для
звичайного автоенкодера — і тоді друга половина втрати просто зникає.

Одна деталь про масштаб, яку часто ховають. Похибка відновлення тут — **середнє**
по 784 пікселях, а розбіжність KL — **сума** по восьми вимірах латента. Щоб β
означало те саме, що в статтях, KL ділиться на 784: тоді обидві половини стають
«на один піксель». Без цієї домовленості число β не означає нічого, і порівнювати
його з чужою статтею не можна.

In [ ]:
def train_model(kind, beta, seed, epochs=25, latent=8, batch=128, lr=3e-3):
    # kind = 'ae' або 'vae'. beta діє лише для 'vae'.
    torch.manual_seed(seed)
    model = Autoencoder(latent) if kind == "ae" else VariationalAutoencoder(latent)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    shuffler = torch.Generator().manual_seed(seed)

    for _ in range(epochs):
        order = torch.randperm(x_train.shape[0], generator=shuffler)
        for start in range(0, x_train.shape[0], batch):
            batch_images = x_train[order[start:start + batch]]

            if kind == "ae":
                restored = model.dec(model.enc(batch_images))
                loss = F.mse_loss(restored, batch_images)
            else:
                mu, logvar = model.encode(batch_images)
                # ось уся репараметризація: випадковість входить окремим доданком
                z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
                restored = model.dec(z)
                kl = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(1)).mean()
                loss = F.mse_loss(restored, batch_images) + beta * kl / 784.0

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    return model


@torch.no_grad()
def reconstruction_error(model, images):
    # Похибка відновлення: подали зображення — отримали його ж назад.
    return F.mse_loss(model.dec(model.code(images)), images).item()


@torch.no_grad()
def sample_from_prior(model, count, seed):
    # Тягнемо z із N(0, 1) і питаємо декодер, що це.
    torch.manual_seed(seed)
    return model.dec(torch.randn(count, model.latent))


print("готово: train_model, reconstruction_error, sample_from_prior")

## 7 · Головний замір: чотири моделі, три зерна

Чотири конфігурації з **однаковим** кістяком і латентом 8: автоенкодер і три VAE з
β = 0.2, 1 і 4. Кожна навчається на трьох зернах, бо різниця, менша за розкид по
зернах, не є різницею.

Для кожної рахуємо пʼять чисел: похибку відновлення, впевненість судді, покриття
класів і обидві відстані. Плюс службове — розмах кодів: наскільки далеко від нуля
модель розкладає свої латентні числа.

In [ ]:
configurations = [("ae", 0.0), ("vae", 0.2), ("vae", 1.0), ("vae", 4.0)]
results = {}
trained = {}

for kind, beta in configurations:
    rows = []
    for seed in [0, 1, 2]:
        started = time.perf_counter()
        model = train_model(kind, beta, seed=seed)
        if seed == 0:
            trained[(kind, beta)] = model     # знадобиться далі для картинок і карти

        row = measure(sample_from_prior(model, 600, seed=100 + seed))
        row["rec"] = reconstruction_error(model, x_test)
        with torch.no_grad():
            codes = model.code(x_test)
            row["code_std"] = codes.std(0).mean().item()
            row["code_norm"] = codes.norm(dim=1).mean().item()
        rows.append(row)
        print("  %-3s beta=%.1f зерно %d: відновлення %.5f · впевненість %.4f · "
              "класів %d · зразок->спр. %.4f · спр.->зразок %.4f  (%.0f с)"
              % (kind, beta, seed, row["rec"], row["conf"], row["cov"],
                 row["near_gen"], row["near_real"], time.perf_counter() - started))
    results["%s_%.1f" % (kind, beta)] = rows

In [ ]:
def mean_of(rows, key):
    return float(np.mean([r[key] for r in rows]))


def spread_of(rows, key):
    return float(np.std([r[key] for r in rows]))


LABELS = {"ae_0.0": "автоенкодер", "vae_0.2": "VAE beta=0.2",
          "vae_1.0": "VAE beta=1", "vae_4.0": "VAE beta=4"}

print("%-14s %12s %16s %8s %14s %15s" %
      ("", "відновлення", "впевненість", "класів", "зразок->спр.", "спр.->зразок"))
print("-" * 84)
for key, rows in results.items():
    print("%-14s %12.5f %9.4f±%.4f %8.2f %14.4f %15.4f" %
          (LABELS[key], mean_of(rows, "rec"),
           mean_of(rows, "conf"), spread_of(rows, "conf"),
           mean_of(rows, "cov"), mean_of(rows, "near_gen"), mean_of(rows, "near_real")))
print()
print("покриття по зернах окремо (купи важливіші за середні):")
for key, rows in results.items():
    print("  %-14s %s" % (LABELS[key], [r["cov"] for r in rows]))
print()
print("розмах кодів (наскільки далеко від нуля лежать латентні числа):")
for key, rows in results.items():
    print("  %-14s std %.2f · середня довжина вектора %.2f"
          % (LABELS[key], mean_of(rows, "code_std"), mean_of(rows, "code_norm")))

Тепер закриємо вимогу калібрування до кінця. Раніше ми перевіряли зіпсовані набори
проти **справжніх даних**. Але правильна планка інша: зіпсований набір мусить
стояти гірше за **робочу модель** — інакше метрика не відрізняє поломку від роботи.

In [ ]:
best_key = "vae_0.2"
best_cov = mean_of(results[best_key], "cov")
best_far = mean_of(results[best_key], "near_real")
print("найкраща робоча модель (%s): класів %.2f, справжня->зразок %.4f"
      % (LABELS[best_key], best_cov, best_far))
print()
for name in ("лише кола", "розмита каша"):
    row = next(r for r in calibration if r["name"] == name)
    caught = []
    if row["cov"] < best_cov:
        caught.append("покриттям")
    if row["near_real"] > best_far:
        caught.append("зворотною відстанню")
    print("%-14s класів %d (проти %.2f) · справжня->зразок %.4f (проти %.4f) -> %s"
          % (name, row["cov"], best_cov, row["near_real"], best_far,
             "спіймано " + " і ".join(caught) if caught else "НЕ СПІЙМАНО"))
print()
weak_cov = mean_of(results["ae_0.0"], "cov")
weak_far = mean_of(results["ae_0.0"], "near_real")
mush = next(r for r in calibration if r["name"] == "розмита каша")
print("а тепер те саме проти автоенкодера: класів %.2f, справжня->зразок %.4f"
      % (weak_cov, weak_far))
print("розмита каша  класів %d, справжня->зразок %.4f" % (mush["cov"], mush["near_real"]))
print()
print("Каша стоїть НЕ гірше за автоенкодер, а трохи краще. Це не поломка метрики,")
print("а її відповідь: розмиті справжні фігури ближчі до справжніх фігур,")
print("ніж те, що автоенкодер малює з випадкового z. Так воно і є.")

## 8 · Суддя різної гостроти: впевненість є властивістю судді, а не зразків

Тепер доведемо, що впевненість не просто «слабка метрика», а взагалі не міряє те,
що ми хочемо. Візьмемо ті самі набори зображень і трьох різних суддів: навченого
10, 20 і 40 епох. Зображення не змінюються **жодним чином** — змінюється тільки те,
хто на них дивиться.

In [ ]:
watch_sets = [
    ("справжні", x_test),
    ("автоенкодер", sample_from_prior(trained[("ae", 0.0)], 600, seed=100)),
    ("VAE beta=0.2", sample_from_prior(trained[("vae", 0.2)], 600, seed=100)),
    ("лише кола", only_circles),
    ("чистий шум", pure_noise),
]

judge_table = []
print("%-6s %9s %s" % ("епох", "точність",
                       " ".join("%13s" % name for name, _ in watch_sets)))
print("-" * 84)
for epochs in [10, 20, 40]:
    other_judge = judge if epochs == 40 else train_judge(epochs=epochs)
    with torch.no_grad():
        accuracy = (other_judge(x_test).argmax(1) == y_test).float().mean().item()
        row = {"epochs": epochs, "acc": accuracy}
        for name, images in watch_sets:
            row[name] = F.softmax(other_judge(images), 1).max(1).values.mean().item()
    judge_table.append(row)
    print("%-6d %9.4f %s" % (epochs, accuracy,
                             " ".join("%13.4f" % row[name] for name, _ in watch_sets)))

## 9 · Ширина вузького місця

Автоенкодер працює тому, що дані мусять пролізти крізь малий шар. Питання, на яке
відповідає замір: **скільки чисел потрібно нашим фігурам** і де ширший латент
перестає допомагати.

Навчаємо звичайні автоенкодери з латентом 1, 2, 4, 8, 16 і 32 — по три зерна на
кожну ширину. Тут беремо **12 епох**, а не 25: нас цікавить форма кривої, а не
рекорд, і вісімнадцять повних навчань зробили б зошит удвічі довшим.

In [ ]:
latent_sizes = [1, 2, 4, 8, 16, 32]
latent_results = {}
narrow_models = {}

for size in latent_sizes:
    errors = []
    for seed in [0, 1, 2]:
        model = train_model("ae", 0.0, seed=seed, epochs=12, latent=size)
        errors.append(reconstruction_error(model, x_test))
        if seed == 0:
            narrow_models[size] = model
    latent_results[size] = errors
    print("латент %2d: відновлення %.5f ±%.5f   (по зернах: %s)"
          % (size, np.mean(errors), np.std(errors),
             " ".join("%.5f" % e for e in errors)))

print()
print("що дає кожне подвоєння:")
previous = None
for size in latent_sizes:
    current = float(np.mean(latent_results[size]))
    if previous is not None:
        print("  %2d -> %2d: похибка впала на %.1f %%"
              % (size // 2, size, 100 * (previous - current) / previous))
    previous = current
print()
print("шумова підлога для порівняння: %.5f" % noise_floor)

## 10 · Де ми беремо z — і чому в автоенкодера там порожньо

Тепер найважливіше в темі. Щоб намалювати **нову** картинку, ми не подаємо на вхід
нічого — ми беремо число з повітря і просимо декодер його намалювати. «З повітря»
означає з `N(0, 1)`: кожен із восьми вимірів тягнемо зі стандартного нормального
розподілу.

Питання: а чи лежать там узагалі коди справжніх зображень? Порахуємо руками.
Довжина вектора з восьми стандартних нормальних чисел у середньому близька до
кореня з восьми, тобто до 2.83. Порівняємо це з тим, де насправді лежать коди
наших фігур.

In [ ]:
print("середня довжина вектора z, узятого з N(0,1) у 8 вимірах: %.2f" % np.sqrt(8))
print()
print("%-15s %16s %26s" % ("", "довжина коду", "кодів усередині кулі 5.66"))
print("-" * 60)
for kind, beta in configurations:
    model = trained[(kind, beta)]
    with torch.no_grad():
        lengths = model.code(x_test).norm(dim=1)
        inside = (lengths < 2 * np.sqrt(8)).float().mean().item()
    print("%-15s %16.2f %24.1f %%"
          % (LABELS["%s_%.1f" % (kind, beta)], lengths.mean().item(), 100 * inside))
print()
print("Куля радіуса 5.66 — це подвоєний типовий радіус вибірки з N(0, 1).")
print("Якщо туди не потрапляє майже жодного справжнього коду, то декодер,")
print("якого ми питаємо саме там, у тому місці ніколи не навчався.")

## 11 · Інтерполяція: замір, який не підтвердив підручник

Класична обіцянка: у VAE проміжна точка між двома кодами осмислена, а в
автоенкодері — ні. Перевіримо це числом, а не оком.

Беремо 40 пар тестових зображень **різних класів**, зʼєднуємо їхні коди відрізком і
йдемо по ньому девʼятьма кроками. У кожній точці міряємо дві речі:

- **розрив** — відстань від точки шляху до найближчого справжнього коду, поділена
  на типову відстань між сусідніми кодами. Ділення обовʼязкове: коди автоенкодера
  живуть на масштабі в кілька разів більшому, і без нормування числа непорівнянні;
- **частку сірих пікселів** у намальованому зображенні — тих, що лежать між 0.2 і
  0.8. Привид із двох фігур, накладених одна на одну, складається саме з таких.

In [ ]:
@torch.no_grad()
def walk_between(model, pairs, steps=9):
    codes = model.code(x_test)
    neighbours = torch.cdist(codes, codes)
    neighbours.fill_diagonal_(float("inf"))
    typical_step = neighbours.min(1).values.mean().item()

    gaps = np.zeros((len(pairs), steps))
    greys = np.zeros((len(pairs), steps))
    for i, (left, right) in enumerate(pairs):
        z_left, z_right = codes[left:left + 1], codes[right:right + 1]
        for k in range(steps):
            t = k / (steps - 1)
            z = z_left * (1 - t) + z_right * t
            gaps[i, k] = torch.cdist(z, codes).min().item() / typical_step
            picture = model.dec(z)
            greys[i, k] = ((picture > 0.2) & (picture < 0.8)).float().mean().item()
    return gaps.mean(0), greys.mean(0), typical_step


chooser = torch.Generator().manual_seed(5)
pairs = []
while len(pairs) < 40:
    left = int(torch.randint(0, 600, (1,), generator=chooser))
    right = int(torch.randint(0, 600, (1,), generator=chooser))
    if y_test[left] != y_test[right]:        # пари беремо лише з різних класів
        pairs.append((left, right))

walks = {}
for key in [("ae", 0.0), ("vae", 0.2)]:
    gap, grey, step = walk_between(trained[key], pairs)
    walks[key] = {"gap": gap, "grey": grey, "step": step}
    label = LABELS["%s_%.1f" % key]
    print("%s: типова відстань між сусідніми кодами %.3f" % (label, step))
    print("   розрив по шляху : %s" % " ".join("%.2f" % v for v in gap))
    print("   сірих пікселів  : %s" % " ".join("%.3f" % v for v in grey))

In [ ]:
middle = 4        # середина шляху з девʼяти точок
print("у середині шляху:")
for key in [("ae", 0.0), ("vae", 0.2)]:
    label = LABELS["%s_%.1f" % key]
    gap, grey = walks[key]["gap"], walks[key]["grey"]
    print("  %-14s розрив %.2f кроку сітки · сірих пікселів %.3f (на кінцях %.3f)"
          % (label, gap[middle], grey[middle], grey[0]))

difference = abs(walks[("ae", 0.0)]["gap"][middle] - walks[("vae", 0.2)]["gap"][middle])
print()
print("різниця розривів: %.2f кроку сітки" % difference)
print("Це менше за десяту частку кроку. Відрізок між двома справжніми кодами")
print("однаково порожній в обох моделях — підручникова теза тут НЕ підтвердилась.")

## 12 · Карта латентного простору

Стиснемо вісім вимірів у два методом головних компонент і подивимось, що там
групується. Міра — точність найближчих сусідів: беремо кожну точку й дивимось,
якого класу пʼятеро її сусідів. Міток при навчанні не було **жодних**, тому все, що
ми побачимо, мережа знайшла сама.

І одразу друге питання, яке зазвичай не ставлять: а чи клас — це справді те, що
латент кодує найкраще? Перевіримо на суперникові: спробуємо вгадати по коду, у якій
**чверті кадру** стоїть фігура.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

# «де стоїть фігура» — чверть кадру за центром маси
pixels = x_test[:, 0].numpy()
rows_grid, cols_grid = np.mgrid[0:28, 0:28]
total = pixels.sum((1, 2)) + 1e-9
center_row = (pixels * rows_grid).sum((1, 2)) / total
center_col = (pixels * cols_grid).sum((1, 2)) / total
quadrant = (center_row > 13.5).astype(int) * 2 + (center_col > 13.5).astype(int)

maps = {}
for key in [("ae", 0.0), ("vae", 0.2)]:
    with torch.no_grad():
        codes = trained[key].code(x_test).numpy()
    projection = PCA(n_components=2, random_state=0)
    flat = projection.fit_transform(codes)

    by_class_2d = KNeighborsClassifier(5).fit(flat, y_test.numpy()).score(flat, y_test.numpy())
    by_class_full = KNeighborsClassifier(5).fit(codes, y_test.numpy()).score(codes, y_test.numpy())
    by_place = KNeighborsClassifier(5).fit(codes, quadrant).score(codes, quadrant)
    maps[key] = {"xy": flat, "class2d": by_class_2d,
                 "classfull": by_class_full, "place": by_place,
                 "var": projection.explained_variance_ratio_}
    print("%-14s клас у 2D %.4f · клас у всіх 8 вимірах %.4f · чверть кадру %.4f"
          % (LABELS["%s_%.1f" % key], by_class_2d, by_class_full, by_place))
    print("%-14s перші дві компоненти тримають %.1f %% дисперсії"
          % ("", 100 * projection.explained_variance_ratio_[:2].sum()))
print()
print("рівень вгадування: клас %.4f, чверть кадру %.4f" % (1 / 6, 1 / 4))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for axis, key in zip(axes, [("ae", 0.0), ("vae", 0.2)]):
    flat = maps[key]["xy"]
    for kind in range(6):
        mask = y_test.numpy() == kind
        axis.scatter(flat[mask, 0], flat[mask, 1], s=9, label=SHAPE_NAMES[kind])
    axis.set_title("%s · сусіди дають клас %.2f"
                   % (LABELS["%s_%.1f" % key], maps[key]["class2d"]), fontsize=10)
    axis.set_xlabel("перша головна компонента")
    axis.set_ylabel("друга головна компонента")
axes[1].legend(fontsize=7, loc="upper right")
plt.tight_layout()
plt.show()
print("кольори — справжні класи; мережа їх не бачила жодного разу")

## 13 · Репараметризація: чому не можна просто насемплити

Лишилось найтонше місце теми. У VAE енкодер видає центр хмарки `mu` й ширину
`sigma`, а декодерові треба **одна точка** з цієї хмарки. Здавалося б: візьми та й
насемпли.

Не вийде — і зараз ми побачимо, чому саме, двома окремими доказами.

**Перший: наївний спосіб рве граф обчислень, а вбудований `torch.normal` мовчки
дає нуль.** Порівняємо чотири способи дістати точку з хмарки. Перший — витягнути з
тензора число й насемплити збоку — те, що пише кожен, хто ще не думав про градієнт.
Другий — вбудований `torch.normal`, який виглядає бездоганно. Третій — наша формула
руками. Четвертий — та сама формула, але вже під іменем бібліотеки.

In [ ]:
mu = torch.tensor(1.5, requires_grad=True)
sigma = torch.tensor(0.7)

def show(name, tensor):
    # чи є граф обчислень і чи доходить по ньому похідна до mu
    if tensor.grad_fn is None:
        print("%-26s графа немає             похідна по mu не існує" % name)
        return
    gradient = torch.autograd.grad(tensor, mu, retain_graph=True)[0].item()
    print("%-26s граф %-18s похідна по mu = %.1f"
          % (name, type(tensor.grad_fn).__name__, gradient))


# спосіб перший, наївний: витягли число, насемплили збоку, повернули тензор
side_rng = np.random.default_rng(0)
show("насемплили збоку", torch.tensor(float(side_rng.normal(mu.item(), 0.7))))

# спосіб другий, найпідступніший: вбудований torch.normal
show("torch.normal(mu, sigma)", torch.normal(mu, sigma))

# спосіб третій: репараметризація вручну
torch.manual_seed(1)
show("mu + sigma * eps", mu + sigma * torch.randn(1))

# спосіб четвертий: те саме, але іменем бібліотеки
cloud = torch.distributions.Normal(mu, sigma)
show("Normal(...).rsample()", cloud.rsample())
print("%-26s %s" % ("Normal(...).sample()",
                    "requires_grad = %s, тобто теж глухий кут" % cloud.sample().requires_grad))

Читай другий рядок уважно. `torch.normal` **має** граф обчислень — і саме тому
виглядає правильно. Але похідна по `mu` в нього дорівнює **нулю**: градієнт
проходить крізь цей вузол і не приносить нічого. Це найгірший із можливих варіантів
відмови — код працює, навчання йде, енкодер не вчиться, і жодного повідомлення
немає. Робоче імʼя цієї операції в PyTorch — `rsample`, де `r` означає рівно
`reparameterized`; звичайний `sample` градієнта не має взагалі.

**Другий доказ: обхідний шлях існує, але він у рази шумніший.**

Похідну від середнього по випадковій величині можна порахувати й без
репараметризації — оцінкою через логарифм щільності. Вона незміщена, тобто в
середньому правильна. Але подивимось на розкид.

Візьмемо задачу, де відповідь відома точно. Нехай `z` тягнеться з нормального
розподілу із середнім `mu` і відхиленням `sigma`, і нас цікавить, як змінюється
середнє від `z` у квадраті, коли ми ворушимо `mu`. Середнє від `z` у квадраті
дорівнює `mu² + sigma²`, отже похідна по `mu` дорівнює рівно `2·mu`. При
`mu = 1.5` це **3.0** — і це число ми знаємо без жодного експерименту.

In [ ]:
mu = torch.tensor(1.5, requires_grad=True)
sigma = torch.tensor(0.7)
samples = 4000
torch.manual_seed(11)

# спосіб репараметризації: z = mu + sigma·eps, похідна d(z²)/d(mu) = 2z
epsilon = torch.randn(samples)
z_reparametrised = mu + sigma * epsilon
per_sample_reparametrised = 2 * z_reparametrised.detach()

# обхідний шлях: та сама похідна через логарифм щільності
z_plain = torch.normal(mu.detach().expand(samples), sigma)
per_sample_score = ((z_plain - mu.detach()) / sigma ** 2) * z_plain ** 2

print("точна відповідь          : %.4f" % 3.0)
print("через mu + sigma·eps     : %.4f ± %.4f"
      % (per_sample_reparametrised.mean(), per_sample_reparametrised.std()))
print("через логарифм щільності : %.4f ± %.4f"
      % (per_sample_score.mean(), per_sample_score.std()))
print()
print("обидві в середньому правильні, але розкид більший у %.1f раза"
      % (per_sample_score.std() / per_sample_reparametrised.std()))

## 14 · Картинки для лекції

Остання службова клітинка: витягуємо самі зображення, які лекція показує в
інтерактивах, і друкуємо їх у стислому вигляді. Кожен піксель — один символ із
шістнадцяти рівнів сірого. Так числа лекції й картинки лекції походять з одного
прогону, а не з двох різних.

In [ ]:
ALPHABET = "0123456789abcdef"


def pack(picture):
    # 28 на 28 значень 0..1 -> рядок із 784 символів, по 16 рівнів сірого
    levels = np.clip(picture.reshape(-1) * 15.999, 0, 15).astype(int)
    return "".join(ALPHABET[v] for v in levels)


NICKNAME = {("ae", 0.0): "ae", ("vae", 0.2): "vae02",
            ("vae", 1.0): "vae1", ("vae", 4.0): "vae4"}

export = {}
with torch.no_grad():
    for key, name in NICKNAME.items():
        pictures = sample_from_prior(trained[key], 8, seed=777)
        export[name + "_samples"] = [pack(p[0].numpy()) for p in pictures]

    # одна й та сама фігура, відновлена автоенкодерами різної ширини
    probe_image = x_test[3:4]
    export["probe"] = pack(probe_image[0, 0].numpy())
    narrow_pictures = {}
    for size in latent_sizes:
        model = narrow_models[size]
        narrow_pictures[str(size)] = pack(model.dec(model.code(probe_image))[0, 0].numpy())
    export["narrow"] = narrow_pictures

    # шлях між двома кодами різних класів
    left, right = pairs[0]
    for key, name in [(("ae", 0.0), "ae"), (("vae", 0.2), "vae02")]:
        model = trained[key]
        z_left = model.code(x_test[left:left + 1])
        z_right = model.code(x_test[right:right + 1])
        export[name + "_walk"] = [
            pack(model.dec(z_left * (1 - k / 4) + z_right * (k / 4))[0, 0].numpy())
            for k in range(5)]

    # справжні фігури для порівняння: по одній кожного класу
    export["real"] = [pack(x_test[i, 0].numpy()) for i in range(6)]

# координати карти латента. Крок брати не можна: класи в наборі йдуть по колу
# з періодом 6, і будь-який крок, не взаємно простий із шісткою, лишить у вибірці
# не всі класи. Беремо всі 600 точок.
for key in [("ae", 0.0), ("vae", 0.2)]:
    flat = maps[key]["xy"]
    export[NICKNAME[key] + "_map"] = " ".join(
        "%.1f %.1f %d %d" % (flat[i, 0], flat[i, 1], y_test[i], quadrant[i])
        for i in range(600))

import hashlib

packed = json.dumps(export, sort_keys=True)
print("зображень запаковано:", sum(len(v) if isinstance(v, (list, dict)) else 1
                                   for v in export.values()))
print("розмір рядка        :", len(packed), "символів")
print("відбиток            :", hashlib.md5(packed.encode()).hexdigest()[:16])
print()
print("перший рядок пікселів кола, відновленого латентом 32:")
print("  " + export["narrow"]["32"][:28])
print("саме ці рядки лекція показує в інтерактивах — вони з цього ж прогону")

In [ ]:
elapsed = time.perf_counter() - notebook_started
print("зошит виконався за %.0f с (%.1f хв)" % (elapsed, elapsed / 60))

## Що показали числа

1. **Впевненість судді не є мірою якості генерації.** Біле полотно дістає від нього
   більшу впевненість, ніж справжні фігури, а повний колапс мод — майже таку саму.
   Причина не в поганому судді: у нього немає варіанта «нічого з переліченого».
2. **Одного числа для генерації не існує.** Відстань «зразок -> найближча справжня»
   хвалить колапс, покриття класів сліпе до якості. Працює тільки пара.
3. **Автоенкодер стискає краще, а генерує гірше** — і ці два твердження міряються
   різними числами, які впорядковують моделі протилежно.
4. **Дірка в латентному просторі автоенкодера — це число, а не метафора.** Коди
   лежать на відстані в десятки одиниць від нуля, а ми тягнемо z із кулі радіуса три.
5. **β поводиться як компроміс, доки не стає завеликим**; після цього він псує
   декодер, і покриття падає.
6. **Інтерполяція між двома справжніми кодами однакова в обох моделях.** Класична
   демонстрація «у VAE середина осмислена» на наших даних **не відтворилась**.

## Завдання

**🟢 Рівень 1.** Додай до таблиці калібрування девʼятий набір: справжні фігури,
зсунуті на 5 пікселів убік (`np.roll`). Яка з чотирьох метрик помітить підміну, а
яка ні?

**🟡 Рівень 2.** Знайди β, при якому покриття класів максимальне. Перебери
β = 0.05, 0.1, 0.2, 0.5 на трьох зернах і побудуй криву «β -> покриття». Чи є
максимум усередині проміжку, чи покриття просто спадає?

**🔴 Рівень 3.** Заміни у VAE втрату відновлення з `mse_loss` на
`binary_cross_entropy`. Це змінює масштаб першої половини втрати, отже й значення β.
Знайди, якому β у новій шкалі відповідає β = 0.2 у старій — критерій збігу візьми за
розмахом кодів (`code_std`).